# 01 · Recueil des données

**Objectif** : savoir d'où viennent les données, vérifier qu'elles sont intactes, et décrire chaque colonne avec son **rôle** dans le projet.

**Source** : Zhang et al. (2025), *A high-resolution electric vehicle charging transaction dataset with multidimensional features in China*, Scientific Data. Figshare : https://doi.org/10.6084/m9.figshare.28182251.v3. Licence **CC BY 4.0** : réutilisation libre, avec citation.

**Contexte** : 13 stations exploitées par State Grid Zhejiang à Jiaxing (Chine), décembre 2019 à décembre 2021.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

from evrel.data.load import get_data_dir, load_config, load_sessions

pd.set_option("display.max_columns", 30)
RAW = get_data_dir() / "raw"
RAW

WindowsPath('C:/dev/ev-reliability-data/raw')

## 1. Fichiers reçus et intégrité

Le **md5** est l'empreinte d'un fichier : un seul octet modifié la change complètement. On compare celle de `Dataset.zip` à celle publiée par Figshare.

In [2]:
files = pd.DataFrame(
    [{"fichier": str(p.relative_to(RAW)), "taille_Mo": round(p.stat().st_size / 1e6, 2)}
     for p in sorted(RAW.rglob("*")) if p.is_file()]
)
files

,fichier,taille_Mo
0,Code\charging_frequency.py,0.00
1,Code\comparison_arrival_time.py,0.00
2,Code\comparison_dataset_in_location.py,0.00
3,Code\comparison_power.py,0.00
4,Code\difference_analysis_covid19.py,0.00
5,Code\failure_analysis.py,0.00
6,Code\kmeans.py,0.00
7,Code\location_covid19.py,0.00
8,Code\power_at_13_sites.py,0.00
9,Code\TOU_analysis.py,0.00


In [3]:
expected = load_config()["source"]["md5"]
zip_path = RAW / "Dataset.zip"
if zip_path.exists():
    actual = hashlib.md5(zip_path.read_bytes()).hexdigest()
    print("md5 attendu :", expected)
    print("md5 obtenu  :", actual)
    print("Intègre" if actual == expected else "ATTENTION : fichier différent de la source")
else:
    print("Dataset.zip absent de raw/ : contrôle impossible")

md5 attendu : d2f43f19a3cfd5d364f97225e5f5d3ab
md5 obtenu  : d2f43f19a3cfd5d364f97225e5f5d3ab
Intègre


## 2. Chargement

`load_sessions()` (dans `src/evrel/data/load.py`) gère l'encodage chinois, les tabulations parasites, les noms de colonnes et les types. Ses propriétés sont vérifiées par `tests/test_load.py`.

In [4]:
sessions = load_sessions()
print(f"{len(sessions):,} sessions × {sessions.shape[1]} colonnes")
sessions.head()

441,077 sessions × 24 colonnes


,user_id,post_id,site,district,order_created_at,energy_kwh,electricity_cost,service_charge,amount,amount_paid,start_time,end_time,paid_at,temperature_c,humidity_pct,precipitation_mm,end_cause,is_full_charge,is_user_stop,is_nonsystem_fault,is_system_issue,is_EV_fault,is_other_fault,is_abnormal
0,1,1,Tourist Attraction,Nanhu,2020-12-31 23:35:55,6.24,2.60,3.74,6.34,6.34,2020-12-31 23:35:43,2020-12-31 23:48:08,2020-12-31 23:48:27,-3.2,45,0.0,User stops charging,0,1,0,0,0,0,0
1,2,2,Tourist Attraction,Nanhu,2020-12-31 23:35:17,0.62,0.26,0.37,0.63,0.63,2020-12-31 23:35:19,2020-12-31 23:38:00,2020-12-31 23:38:04,-3.2,45,0.0,User stops charging,0,1,0,0,0,0,0
2,3,3,Tourist Attraction,Nanhu,2020-12-31 22:01:34,28.01,11.66,16.81,28.47,28.47,2020-12-31 22:00:05,2020-12-31 22:39:38,2020-12-31 22:41:14,-3.2,45,0.0,User stops charging,0,1,0,0,0,0,0
3,4,2,Tourist Attraction,Nanhu,2020-12-31 22:21:37,5.20,2.17,3.12,5.29,5.29,2020-12-31 22:21:38,2020-12-31 22:34:08,2020-12-31 22:34:12,-3.2,45,0.0,User stops charging,0,1,0,0,0,0,0
4,5,1,Tourist Attraction,Nanhu,2020-12-31 22:10:30,4.58,1.91,2.75,4.66,4.66,2020-12-31 22:10:31,2020-12-31 22:20:10,2020-12-31 22:20:16,-3.2,45,0.0,Charging ends normally,1,0,0,0,0,0,0


In [5]:
sessions.dtypes.to_frame("type")

,type
user_id,int64
post_id,int64
site,object
district,object
order_created_at,datetime64[ns]
energy_kwh,float64
electricity_cost,float64
service_charge,float64
amount,float64
amount_paid,float64


## 3. Valeurs manquantes et doublons

In [6]:
quality = pd.DataFrame({
    "manquants": sessions.isna().sum(),
    "manquants_%": (sessions.isna().mean() * 100).round(2),
    "valeurs_distinctes": sessions.nunique(),
})
quality

,manquants,manquants_%,valeurs_distinctes
user_id,0,0.0,56115
post_id,0,0.0,92
site,0,0.0,13
district,0,0.0,3
order_created_at,0,0.0,438764
energy_kwh,0,0.0,7053
electricity_cost,0,0.0,6320
service_charge,0,0.0,5129
amount,0,0.0,8487
amount_paid,0,0.0,8412


In [7]:
dup_full = sessions.duplicated().sum()
dup_key = sessions.duplicated(["post_id", "start_time"]).sum()
print(f"Lignes entièrement dupliquées : {dup_full:,}")
print(f"Même borne et même heure de début : {dup_key:,}")

Lignes entièrement dupliquées : 0
Même borne et même heure de début : 0


## 4. Dictionnaire des colonnes

Le **rôle** est la colonne la plus importante : il dit ce qu'on a le droit d'utiliser pour prédire. Une information connue **après** la session (paiement) ne peut pas servir à prédire, sinon c'est une **fuite de données**.

| Colonne | Contenu | Rôle dans le projet |
|---|---|---|
| `user_id` | Identifiant de l'usager | Contexte (diversité des usagers) |
| `post_id` | Borne (1 à 92) | **Clé** de la prédiction |
| `site`, `district` | Station et district | Variable de contexte |
| `order_created_at` | Création de la commande | Horodatage technique |
| `start_time`, `end_time` | Début et fin de la session | Base du temps (jour, durée) |
| `energy_kwh` | Énergie délivrée | Mesure d'usage |
| `electricity_cost`, `service_charge`, `amount` | Montants facturés | Peu utiles (dérivés du tarif et de l'énergie) |
| `amount_paid`, `paid_at` | Paiement | **Connus après coup : jamais en variable** |
| `temperature_c`, `humidity_pct`, `precipitation_mm` | Météo journalière du district | Contexte (celle de la veille) |
| `end_cause` | Cause de fin de session | **Source de la cible** |
| `is_*` (7 indicateurs) | Familles de causes définies par les auteurs | Non utilisés : remplacés par notre propre classement |

## Ce qu'on retient

* Intégrité confirmée : le md5 correspond à celui publié par Figshare. On travaille bien sur les données des auteurs.
* Aucune valeur manquante et aucun doublon, ni ligne entière, ni même borne à la même seconde. C’est rare et appréciable.
* 56 115 usagers pour 441 077 sessions : environ 8 sessions par usager en moyenne. Beaucoup d’usagers occasionnels, donc.
* Les types sont bons : dates en datetime64, indicateurs en int8.
L* e fichier n’est pas trié dans le temps : les premières lignes datent du 31 décembre 2020, alors que les données commencent fin 2019. Tout calcul de type « jours précédents » devra trier d’abord.